# Secondary-structure effect on ¹⁵N accuracy (ignoring solvent exposure) — PANAV test set, 10-seed ensemble (ProteinMPNN (finetuned))

Pairwise protein-cluster bootstrap (Holm), and one-factor contrasts among the six SS × exposure groups. Residues: GT present, prediction mask True, no proline.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal

SS_ORDER = ['helix', 'sheet', 'coil']
EXPOSURE_ORDER = ['buried', 'exposed']
PRED = "../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed_mpnn_ft.json"
GT   = "../../../data/rBMRB/cluster_30p/PANAV/test.json"
PRR  = "../../../data/analysis/Fig4/sasa_results/per_residue_records.json"

In [ ]:
# ---------- build per-residue |error| with ss_class + exposure ----------
pred_data = json.load(open(PRED)); gt_data = json.load(open(GT)); prr_data = json.load(open(PRR))
pred_dict, mask_dict = pred_data['mean'], pred_data['mask']   # 10-seed ensemble mean, not the per-seed 'pred'
dictionary = {v: k for k, v in gt_data['entry_ID'].items()}

rows = []
for entry in gt_data['entry_ID'].values():
    key = dictionary.get(entry)
    if key is None or entry not in pred_dict:
        continue
    prr_entry = prr_data.get(entry)
    if prr_entry is None:
        continue

    gt_seq, gt_shifts = gt_data['sequence'][key], gt_data['N'][key]
    pred_arr_full  = np.array(pred_dict[entry])
    pred_mask_full = np.array(mask_dict[entry], dtype=bool)
    n = len(gt_seq)
    # Prediction arrays are padded to the longest sequence in the batch; any tail beyond gt_seq is
    # mask=False / mean=None padding, so truncate to n instead of dropping the protein
    if len(pred_arr_full) < n or len(pred_mask_full) < n:
        continue
    pred_arr = pred_arr_full[:n]
    pred_mask = pred_mask_full[:n]

    prr_lookup = {r['resnum']: r for r in prr_entry}

    # Proline / None exclusion + the prediction's own mask (tag/flag and outlier exclusion already included)
    gt_mask_1 = np.array([res != 'P' for res in gt_seq])
    none_mask = np.array([g is not None for g in gt_shifts])
    mask = gt_mask_1 & none_mask & pred_mask
    if mask.sum() == 0:
        continue

    full_indices = np.where(mask)[0]
    pred_f = pred_arr[mask]
    gt_f   = np.array(gt_shifts)[mask].astype(float)
    abs_error = np.abs(pred_f - gt_f)   # no analytical centering (err - err.mean())

    for i, full_idx in enumerate(full_indices):
        resnum = int(full_idx) + 1
        rec = prr_lookup.get(resnum)
        if rec is None or rec.get('ss_class') is None:
            continue
        rows.append(dict(entry=entry, residue_type=gt_seq[full_idx], ss_class=rec['ss_class'],
                          exposure=rec.get('exposure'), abs_error=abs_error[i]))

df = pd.DataFrame(rows)
print(f"df rows: {len(df)} | proteins: {df.entry.nunique()}")
print("per-SS counts:", {s: int((df.ss_class==s).sum()) for s in SS_ORDER})
print("per-exposure counts:", {e: int((df.exposure==e).sum()) for e in EXPOSURE_ORDER})

In [ ]:
from pathlib import Path

B_BOOT = 20000      # bootstrap resamples per contrast; 2/B_BOOT x n_tests must stay below the smallest star cut
RECOMPUTE = False   # True -> ignore cached csv files and rerun the bootstrap (e.g. after GT / prediction updates)


def cached(path, compute):
    """Load a bootstrap result table from csv if it exists with the same B_BOOT, otherwise compute and save it."""
    path = Path(path)
    if path.exists() and not RECOMPUTE:
        R = pd.read_csv(path)
        if "B" in R.columns and (R.B == B_BOOT).all():
            print(f"loaded {path} (B={B_BOOT})")
            return R
        print(f"{path} has a different B -> recomputing")
    R = compute()
    R["B"] = B_BOOT
    R.to_csv(path, index=False)
    print(f"saved {path} (B={B_BOOT})")
    return R

# ---------- protein-cluster bootstrap + Holm ----------
def boot_delta(sub, gcol, gA, gB, B=B_BOOT, seed=0):
    """median(gB)-median(gA) with proteins resampled with replacement -> respects within-protein correlation."""
    rng = np.random.default_rng(seed)
    ent = sub.entry.to_numpy(); grp = sub[gcol].to_numpy(); val = sub.abs_error.to_numpy()
    uniq = np.unique(ent); pidx = {e: np.where(ent==e)[0] for e in uniq}
    obs = np.median(val[grp==gB]) - np.median(val[grp==gA])
    d = []
    for _ in range(B):
        samp = rng.choice(uniq, uniq.size, replace=True)
        idx = np.concatenate([pidx[e] for e in samp])
        a = val[idx][grp[idx]==gA]; b = val[idx][grp[idx]==gB]
        if len(a) < 5 or len(b) < 5:
            continue
        d.append(np.median(b) - np.median(a))
    d = np.array(d)
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return obs, lo, hi, p

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

## One-factor contrasts (SS within exposure, exposure within SS)

In [ ]:
# ---------- TASK 3: SS-vs-SS (within exposure) + exposure-vs-exposure (within SS) ----------
from itertools import combinations

df2 = df[df.exposure.isin(EXPOSURE_ORDER)].copy()
df2['group'] = df2.ss_class + '_' + df2.exposure
GROUPS = [f"{ss}_{e}" for ss in SS_ORDER for e in EXPOSURE_ORDER]

# only same-factor pairs: SS-vs-SS at fixed exposure, and exposure-vs-exposure at fixed SS
pairs3 = [(f"{a}_{e}", f"{b}_{e}", "SS") for e in EXPOSURE_ORDER for a, b in combinations(SS_ORDER, 2)]
pairs3 += [(f"{ss}_buried", f"{ss}_exposed", "exposure") for ss in SS_ORDER]
def compute_R3():
    rows3 = []
    for a, b, kind in pairs3:
        sub = df2[df2.group.isin([a, b])]
        obs, lo, hi, p = boot_delta(sub, 'group', a, b)
        rows3.append(dict(kind=kind, group_a=a, group_b=b, contrast=f"{b} - {a}", delta=obs, ci_lo=lo, ci_hi=hi,
                          boot_p=p, n_a=int((df2.group==a).sum()), n_b=int((df2.group==b).sum())))
    R = pd.DataFrame(rows3)
    R['p_holm'] = holm(R.boot_p.values)
    R['sig'] = R.p_holm < 0.05
    return R


R3 = cached("../../../data/analysis/Fig4/group_pairwise_panav10seedens_mpnn_ft.csv", compute_R3)
pd.set_option("display.width", 200, "display.max_columns", 20)
print(f"=== SS-vs-SS and exposure-vs-exposure contrasts ({len(R3)} contrasts, no cross terms; "
      f"protein-cluster bootstrap B={B_BOOT}, Holm-corrected) ===")
print(R3.round(4).to_string(index=False))


## Visualization: SS × exposure groups

In [ ]:
import matplotlib.pyplot as plt
def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base, "xtick.labelsize": base-2,
        "ytick.labelsize": base-2, "legend.fontsize": base-2, "svg.fonttype": "none",
        "mathtext.default": "regular"})
set_style(16)

P_FLOOR = 2 / B_BOOT * len(R3)     # smallest Holm-adjusted p the bootstrap can resolve (two-sided -> 2/B)
def stars(p):
    p = max(p, P_FLOOR)
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."

grp_sig = {}
for r in R3.itertuples():
    grp_sig[(r.group_a, r.group_b)] = stars(r.p_holm)
    grp_sig[(r.group_b, r.group_a)] = stars(r.p_holm)
def sig_for_groups(a, b):
    return grp_sig.get((a, b), "n.s.")

# ---------- visualize: |error| by SS within each exposure level ----------
SS_COL2 = {'helix': "#61AFEF", 'sheet': "#E06C75", 'coil': "#98C379"}
EXP_TITLE = {'buried': 'Buried', 'exposed': 'Exposed'}
YCAP3 = 4.0

fig, axes = plt.subplots(1, len(EXPOSURE_ORDER), figsize=(5*len(EXPOSURE_ORDER), 5.5), sharey=True)
for ax, e in zip(axes, EXPOSURE_ORDER):
    sub = df2[df2.exposure == e]
    data = [sub[sub.ss_class == s].abs_error.values for s in SS_ORDER]
    vp = ax.violinplot([np.clip(d, 0, YCAP3) for d in data], showmedians=True, widths=0.85)
    for b, s in zip(vp["bodies"], SS_ORDER):
        b.set_facecolor(SS_COL2[s]); b.set_alpha(0.75); b.set_edgecolor("white")
    for k in ("cbars", "cmins", "cmaxes", "cmedians"):
        vp[k].set_edgecolor("#444"); vp[k].set_linewidth(1.3)
    ax.set_xticks(range(1, len(SS_ORDER) + 1))
    # ax.set_xticklabels([f"{s.capitalize()}\n(med {np.median(d):.2f}\nn={len(d)})" for s, d in zip(SS_ORDER, data)])
    ax.set_xticklabels([f"{s.capitalize()}" for s, d in zip(SS_ORDER, data)])
    ax.set_title(EXP_TITLE[e]); ax.set_ylim(0, YCAP3 + 0.9)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    def bracket(ax, x1, x2, y, label):
        ax.plot([x1, x1, x2, x2], [y, y+0.08, y+0.08, y], lw=1.2, c="k")
        ax.text((x1+x2)/2, y+0.10, label, ha="center", va="bottom", fontsize=14 if label != "n.s." else 14)
    bracket(ax, 1, 2, YCAP3 + 0.05, sig_for_groups(f"helix_{e}", f"sheet_{e}"))
    bracket(ax, 2, 3, YCAP3 + 0.05, sig_for_groups(f"sheet_{e}", f"coil_{e}"))
    bracket(ax, 1, 3, YCAP3 + 0.45, sig_for_groups(f"helix_{e}", f"coil_{e}"))
axes[0].set_ylabel("|error| (ppm)")
fig.suptitle("ProteinMPNN (finetuned)", fontsize=16, y=0.93, font="Arial")
plt.tight_layout()
# fig.savefig("Figs/ss_within_exposure_violin_panav10seedens_mpnn_ft.svg", bbox_inches="tight")
# fig.savefig("Figs/ss_within_exposure_violin_panav10seedens_mpnn_ft.png", dpi=300, bbox_inches="tight")
plt.show()
# print("saved ss_within_exposure_violin_panav10seedens_mpnn_ft.svg/.png")

In [ ]:
def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base, "xtick.labelsize": base-2,
        "ytick.labelsize": base-2, "legend.fontsize": base-4, "svg.fonttype": "none",
        "mathtext.default": "regular"})
set_style(21)

# ---------- split violin: buried (left half) vs exposed (right half) within each SS ----------
from matplotlib.patches import Patch

EXP_SIDE = {"buried": -1, "exposed": 1}         # -1 = left half, +1 = right half
EXP_ALPHA = {"buried": 0.9, "exposed": 0.4}

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for x, s in enumerate(SS_ORDER, start=1):
    for e in ("buried", "exposed"):
        d = df2[(df2.ss_class == s) & (df2.exposure == e)].abs_error.values
        vp = ax.violinplot([np.clip(d, 0, YCAP3)], positions=[x], showextrema=False, widths=0.85)
        body = vp["bodies"][0]
        v = body.get_paths()[0].vertices
        med = np.median(d)
        w = np.abs(v[np.argmin(np.abs(v[:, 1] - med)), 0] - x)   # half-width of the violin at the median
        if EXP_SIDE[e] < 0:                                      # keep one half: cut at the centre line
            v[:, 0] = np.minimum(v[:, 0], x)
        else:
            v[:, 0] = np.maximum(v[:, 0], x)
        body.set_facecolor(SS_COL2[s]); body.set_alpha(EXP_ALPHA[e]); body.set_edgecolor("white")
        ax.plot([x, x + EXP_SIDE[e] * w], [med, med], c="#444", lw=1.3)

# SS-vs-SS contrasts within each exposure level (R3): one bracket per SS pair, label "buried / exposed"
# def bracket(x1, x2, y, label):
#     ax.plot([x1, x1, x2, x2], [y, y + 0.08, y + 0.08, y], lw=1.2, c="k")
#     ax.text((x1 + x2) / 2, y + 0.10, label, ha="center", va="bottom", fontsize=17)
# for (a, b), y in ((("helix", "sheet"), YCAP3 + 0.05), (("sheet", "coil"), YCAP3 + 0.05),
#                   (("helix", "coil"), YCAP3 + 0.55)):
#     x1, x2 = SS_ORDER.index(a) + 1, SS_ORDER.index(b) + 1
#     lab = " / ".join(sig_for_groups(f"{a}_{e}", f"{b}_{e}") for e in ("buried", "exposed"))
#     bracket(x1 + 0.03 if x1 == 2 else x1, x2 - 0.03 if x2 == 2 else x2, y, lab)   # small gap at Sheet

# buried vs exposed within each SS (exposure contrasts in R3): one bracket over each split violin
def bracket(x1, x2, y, label):
    ax.plot([x1, x1, x2, x2], [y, y + 0.08, y + 0.08, y], lw=1.2, c="k")
    ax.text((x1 + x2) / 2, y + 0.10, label, ha="center", va="bottom", fontsize=17)
for x, s in enumerate(SS_ORDER, start=1):
    bracket(x - 0.2, x + 0.2, YCAP3 + 0.05, sig_for_groups(f"{s}_buried", f"{s}_exposed"))

ax.set_xticks(range(1, len(SS_ORDER) + 1))
SS_ORDER_up = ['helix', 'sheet', 'loop']
ax.set_xticklabels([s.capitalize() for s in SS_ORDER_up])
ax.set_ylim(0, YCAP3 + 1.1)
ax.set_ylabel("|error| (ppm)")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(handles=[Patch(facecolor="grey", alpha=EXP_ALPHA["buried"], label="Buried (left)"),
                   Patch(facecolor="grey", alpha=EXP_ALPHA["exposed"], label="Exposed (right)")],
          frameon=False, loc="lower center", bbox_to_anchor=(0.5, 0.95), ncol=2,
          prop={"family": "Arial", "size": 18})                                   # legend entries
        #   title="Significance brackets: buried / exposed",
        #   title_fontproperties={"family": "Arial", "size": 15, "style": "italic"})  # title
ax.set_title("ProteinMPNN (finetuned)", fontsize=19, pad=25)
plt.tight_layout()
fig.savefig("Figs/ss_exposure_split_violin_panav10seedens_mpnn_ft.svg", bbox_inches="tight")
fig.savefig("Figs/ss_exposure_split_violin_panav10seedens_mpnn_ft.png", dpi=300, bbox_inches="tight")
plt.show()
